# 02 Baselines

**Goal:** build the train, validation and test split from the EDA, set a rule based baseline, and fit a first logistic regression.

Decisions carried over from `01_eda.ipynb`:
- Split inside 1 to 10 Sep 2022, with the 11 to 18 Sep tail held apart.
- `self_transfer` is a feature. Payment format and payment currency are ablated.
- Metrics are PR-AUC and recall at a fixed false positive rate.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")

TX_COLS = [
    "timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_received", "receiving_currency", "amount_paid",
    "payment_currency", "payment_format", "is_laundering",
]
TX_DTYPES = {
    "from_bank": "int32", "from_account": "category",
    "to_bank": "int32", "to_account": "category",
    "amount_received": "float64", "receiving_currency": "category",
    "amount_paid": "float64", "payment_currency": "category",
    "payment_format": "category", "is_laundering": "int8",
}


def load_transactions(path):
    tx = pd.read_csv(path, header=0, names=TX_COLS, dtype=TX_DTYPES)
    tx["timestamp"] = pd.to_datetime(tx["timestamp"], format="%Y/%m/%d %H:%M")
    tx["date"] = tx["timestamp"].dt.normalize()
    tx["self_transfer"] = (
        (tx["from_account"].astype(str) == tx["to_account"].astype(str))
        & (tx["from_bank"] == tx["to_bank"])
    )
    return tx


tx = load_transactions(DATA / "HI-Small_Trans.csv")
print(f"rows: {len(tx):,}")

rows: 5,078,345


In [2]:
SPLITS = {
    "train": ("2022-09-01", "2022-09-06"),
    "validation": ("2022-09-07", "2022-09-08"),
    "test": ("2022-09-09", "2022-09-10"),
}


def split_by_date(tx, splits=SPLITS):
    return {
        name: tx[(tx["date"] >= start) & (tx["date"] <= end)]
        for name, (start, end) in splits.items()
    }


parts = split_by_date(tx)
train, val, test = parts["train"], parts["validation"], parts["test"]

for name, part in parts.items():
    print(f"{name:<11} rows: {len(part):>9,}   laundering: {part['is_laundering'].sum():>5,}")

train       rows: 3,248,921   laundering: 2,530
validation  rows:   965,524   laundering: 1,036
test        rows:   862,792   laundering:   956


## Evaluation

Every model is scored the same way on the validation set:
- **PR-AUC** (average precision), the main metric at this class balance.
- **Recall at a fixed false positive rate** of 0.1% and 1%, which answers "how much laundering is caught if analysts can only review this many alerts".

A model that ranks at random has a PR-AUC equal to the laundering rate, about 0.001.

In [3]:
from sklearn.metrics import average_precision_score, roc_curve


def recall_at_fpr(y_true, scores, max_fpr):
    fpr, tpr, _ = roc_curve(y_true, scores)
    return float(tpr[fpr <= max_fpr].max())


def evaluate(y_true, scores, name, fprs=(0.001, 0.01)):
    result = {"model": name, "pr_auc": average_precision_score(y_true, scores)}
    for f in fprs:
        result[f"recall_at_{f:.1%}_fpr"] = recall_at_fpr(y_true, scores, f)
    return result


results = []
y_val = val["is_laundering"].to_numpy()
print(f"validation laundering rate: {y_val.mean():.4%}")

validation laundering rate: 0.1073%


In [4]:
ach_flag = (val["payment_format"] == "ACH").to_numpy().astype(int)

tp = int(((ach_flag == 1) & (y_val == 1)).sum())
fp = int(((ach_flag == 1) & (y_val == 0)).sum())
fn = int(((ach_flag == 0) & (y_val == 1)).sum())
tn = int(((ach_flag == 0) & (y_val == 0)).sum())

print(f"alerts raised: {tp + fp:,} of {len(val):,} transactions")
print(f"precision: {tp / (tp + fp):.2%}")
print(f"recall: {tp / (tp + fn):.2%}")
print(f"false positive rate: {fp / (fp + tn):.2%}")

results.append(evaluate(y_val, ach_flag, "ACH rule"))
pd.DataFrame(results)

alerts raised: 116,910 of 965,524 transactions
precision: 0.76%
recall: 85.91%
false positive rate: 12.03%


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,ACH rule,0.006691,0.0,0.0


## Features and logistic regression

First model: logistic regression on transaction level features only. It is fitted twice, without and with the two shortcut columns (payment format and payment currency), to see how much of the score depends on them.

In [5]:
currency_median = (
    train.groupby("payment_currency", observed=True)["amount_paid"].median().to_dict()
)


def add_features(df, currency_median):
    out = pd.DataFrame(index=df.index)
    out["log_amount"] = np.log10(df["amount_paid"].clip(lower=0.01))
    typical = df["payment_currency"].astype(str).map(currency_median)
    out["amount_vs_currency"] = out["log_amount"] - np.log10(typical)
    hour = df["timestamp"].dt.hour
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    out["self_transfer"] = df["self_transfer"].astype(int)
    out["payment_format"] = df["payment_format"].astype(str)
    out["payment_currency"] = df["payment_currency"].astype(str)
    return out


X_train = add_features(train, currency_median)
X_val = add_features(val, currency_median)
y_train = train["is_laundering"].to_numpy()

X_train.head()

,log_amount,amount_vs_currency,hour_sin,hour_cos,self_transfer,payment_format,payment_currency
0,3.567889,0.573775,0.0,1.0,1,Reinvestment,US Dollar
1,-2.000000,-4.994115,0.0,1.0,0,Cheque,US Dollar
2,4.166595,1.172480,0.0,1.0,1,Reinvestment,US Dollar
3,3.448238,0.454123,0.0,1.0,1,Reinvestment,US Dollar
4,4.564464,1.570350,0.0,1.0,1,Reinvestment,US Dollar


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ["log_amount", "amount_vs_currency", "hour_sin", "hour_cos", "self_transfer"]
SHORTCUTS = ["payment_format", "payment_currency"]


def fit_logreg(X, y, categorical):
    steps = [("num", StandardScaler(), NUMERIC)]
    if categorical:
        steps.append(("cat", OneHotEncoder(handle_unknown="ignore"), categorical))
    model = make_pipeline(
        ColumnTransformer(steps),
        LogisticRegression(class_weight="balanced", max_iter=500),
    )
    return model.fit(X, y)


models = {}
for name, categorical in [
    ("LogReg, no shortcut columns", []),
    ("LogReg, with format and currency", SHORTCUTS),
]:
    models[name] = fit_logreg(X_train, y_train, categorical)
    scores = models[name].predict_proba(X_val)[:, 1]
    results.append(evaluate(y_val, scores, name))

pd.DataFrame(results)

,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,ACH rule,0.006691,0.000000,0.000000
1,"LogReg, no shortcut columns",0.002210,0.001931,0.018340
2,"LogReg, with format and currency",0.016349,0.031853,0.131274


**Finding:** Logistic regression on transaction level features alone reaches a PR-AUC of 0.0022, about twice a random ranking and below the ACH rule (0.0067). At a 1% false positive rate it catches 19 of 1,036 laundering transactions. Adding payment format and payment currency raises PR-AUC to 0.0163 and recall at a 1% false positive rate to 136 of 1,036, so almost all of the model's skill comes from the two shortcut columns. A single transaction viewed in isolation carries little signal, which fits the multi-hop structure seen in the patterns file. The linear model also cannot represent the amount band found in the EDA, where laundering is concentrated between roughly 1,000 and 20,000.

## Gradient boosting

Same features as the logistic regression, with a non-linear model. This tests whether the weak result without shortcut columns was a limit of the linear model or of the features.

LightGBM is trained with L2 regularisation and early stopping on validation PR-AUC. An earlier run without them, fixed at 300 trees, was unstable at this class balance: validation PR-AUC peaked near 100 trees and then fell, and single late trees produced extreme gain values.

In [7]:
import lightgbm as lgb

CATEGORIES = {c: sorted(X_train[c].unique()) for c in SHORTCUTS}


def gbm_frame(X, use_shortcuts, numeric=NUMERIC):
    cols = numeric + (SHORTCUTS if use_shortcuts else [])
    out = X[cols].copy()
    for c in SHORTCUTS:
        if c in out:
            out[c] = pd.Categorical(out[c], categories=CATEGORIES[c])
    return out


def fit_gbm(X_tr, y_tr, X_va, y_va, use_shortcuts, numeric=NUMERIC, seed=42):
    model = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=200,
        reg_lambda=10.0,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        metric="average_precision",
        random_state=seed,
        verbose=-1,
    )
    model.fit(
        gbm_frame(X_tr, use_shortcuts, numeric),
        y_tr,
        eval_set=[(gbm_frame(X_va, use_shortcuts, numeric), y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    return model


def run_gbm(name, X_tr, X_va, use_shortcuts, numeric=NUMERIC):
    model = fit_gbm(X_tr, y_train, X_va, y_val, use_shortcuts, numeric)
    scores = model.predict_proba(gbm_frame(X_va, use_shortcuts, numeric))[:, 1]
    models[name] = model
    results.append({**evaluate(y_val, scores, name), "trees": model.best_iteration_})


run_gbm("LightGBM, no shortcut columns", X_train, X_val, False)
run_gbm("LightGBM, with format and currency", X_train, X_val, True)

pd.DataFrame(results)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr,trees
0,ACH rule,0.006691,0.000000,0.000000,NaN
1,"LogReg, no shortcut columns",0.002210,0.001931,0.018340,NaN
2,"LogReg, with format and currency",0.016349,0.031853,0.131274,NaN
3,"LightGBM, no shortcut columns",0.006639,0.015444,0.122587,49.0
4,"LightGBM, with format and currency",0.075553,0.157336,0.500965,95.0


**Finding:** With the same features, LightGBM beats logistic regression. Without the shortcut columns, recall at a 1% false positive rate rises from 19 to 127 of 1,036 laundering transactions, which supports the view that the linear model could not represent the amount band seen in the EDA. PR-AUC without shortcut columns is still only 0.0066, level with the ACH rule (0.0067), so transaction level features remain the main limit. With payment format and payment currency, PR-AUC reaches 0.0756 and the model catches 519 of 1,036 laundering transactions (50%) at a 1% false positive rate, about one true alert in 20, against one in 131 for the ACH rule. The shortcut columns multiply PR-AUC by about eleven, so this model depends heavily on them.

## Graph derived features

Each transaction gets counts describing the recent activity of its sender and receiver. Every count uses only transactions with an earlier timestamp, excludes self transfers, and covers a trailing window of 1 or 3 days.

In [8]:
BIG = 100_000
WINDOWS = {"1d": 1440, "3d": 4320}  # minutes


def account_codes(tx):
    from_key = tx["from_bank"].astype(str) + "_" + tx["from_account"].astype(str)
    to_key = tx["to_bank"].astype(str) + "_" + tx["to_account"].astype(str)
    codes, _ = pd.factorize(pd.concat([from_key, to_key], ignore_index=True))
    return codes[: len(tx)], codes[len(tx):]


def count_prior(event_codes, event_minutes, query_codes, query_minutes, window):
    events = np.sort(event_codes.astype(np.int64) * BIG + event_minutes)
    q = query_codes.astype(np.int64) * BIG + query_minutes
    return np.searchsorted(events, q, "left") - np.searchsorted(events, q - window, "left")


def add_graph_features(tx, windows=WINDOWS):
    from_code, to_code = account_codes(tx)
    minutes = (
        (tx["timestamp"] - tx["timestamp"].min()).dt.total_seconds() // 60
    ).to_numpy().astype(np.int64)

    real = ~tx["self_transfer"].to_numpy()
    ev_from, ev_to, ev_min = from_code[real], to_code[real], minutes[real]

    out = pd.DataFrame(index=tx.index)
    for label, w in windows.items():
        out[f"sender_out_{label}"] = count_prior(ev_from, ev_min, from_code, minutes, w)
        out[f"sender_in_{label}"] = count_prior(ev_to, ev_min, from_code, minutes, w)
        out[f"receiver_in_{label}"] = count_prior(ev_to, ev_min, to_code, minutes, w)
        out[f"receiver_out_{label}"] = count_prior(ev_from, ev_min, to_code, minutes, w)

    n_codes = int(max(from_code.max(), to_code.max())) + 1
    _, pair_code = np.unique(from_code.astype(np.int64) * n_codes + to_code, return_inverse=True)
    out["pair_prior_3d"] = count_prior(pair_code[real], ev_min, pair_code, minutes, WINDOWS["3d"])
    return out


G = add_graph_features(tx)
GRAPH = list(G.columns)
G.describe().T[["mean", "50%", "max"]]

,mean,50%,max
sender_out_1d,849.619720,3.0,26368.0
sender_in_1d,6.574134,1.0,539.0
receiver_in_1d,2.407124,2.0,538.0
receiver_out_1d,8.385593,0.0,26359.0
sender_out_3d,2052.326746,8.0,59868.0
sender_in_3d,12.369016,3.0,545.0
receiver_in_3d,5.766074,5.0,544.0
receiver_out_3d,18.100522,0.0,59867.0
pair_prior_3d,3.554806,3.0,24.0


In [9]:
check = tx.join(G)
fan = check[
    (check["from_account"] == "800737690") & (check["is_laundering"] == 1)
].sort_values("timestamp")
fan[["timestamp", "to_account", "sender_out_1d", "sender_out_3d", "is_laundering"]]

,timestamp,to_account,sender_out_1d,sender_out_3d,is_laundering
137520,2022-09-01 00:06:00,80011F990,0,0,1
137522,2022-09-01 04:33:00,80020C5B0,1,1,1
137524,2022-09-01 09:14:00,80006A5E0,2,2,1
137525,2022-09-01 09:56:00,8007A5B70,3,3,1
137526,2022-09-01 11:28:00,80093C0D0,4,4,1
137527,2022-09-01 13:13:00,80078E200,5,5,1
137529,2022-09-01 14:11:00,80066B990,6,6,1
137531,2022-09-02 15:40:00,8002CC310,0,7,1
137532,2022-09-02 21:23:00,8004030A0,1,8,1
137533,2022-09-02 23:10:00,800578800,2,9,1


## LightGBM with graph derived features

Same model and settings as before, with the nine graph derived features added.

In [10]:
ALL_NUMERIC = NUMERIC + GRAPH

X_train_g = X_train.join(G)
X_val_g = X_val.join(G)

run_gbm("LightGBM + graph, no shortcut columns", X_train_g, X_val_g, False, ALL_NUMERIC)
run_gbm("LightGBM + graph, with format and currency", X_train_g, X_val_g, True, ALL_NUMERIC)

pd.DataFrame(results)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr,trees
0,ACH rule,0.006691,0.000000,0.000000,NaN
1,"LogReg, no shortcut columns",0.002210,0.001931,0.018340,NaN
2,"LogReg, with format and currency",0.016349,0.031853,0.131274,NaN
3,"LightGBM, no shortcut columns",0.006639,0.015444,0.122587,49.0
4,"LightGBM, with format and currency",0.075553,0.157336,0.500965,95.0
5,"LightGBM + graph, no shortcut columns",0.270240,0.323359,0.662162,118.0
6,"LightGBM + graph, with format and currency",0.446412,0.508687,0.770270,195.0


**Finding:** Adding nine graph derived features changes the picture. Without the shortcut columns, LightGBM reaches a PR-AUC of 0.2702, about forty times the same model without graph features (0.0066) and more than three times the earlier model that used payment format and currency (0.0756). It catches 686 of 1,036 laundering transactions (66%) at a 1% false positive rate, about one true alert in 15, and 335 (32%) at a 0.1% false positive rate, about one in 4. With the shortcut columns PR-AUC rises to 0.4464 and recall at a 1% false positive rate to 798 (77%), so they now multiply PR-AUC by about 1.7, against about eleven before. Most of the signal is in how accounts behave across transactions. Every count uses only earlier timestamps, which was verified by hand on a known fan-out account. Validation scores are slightly optimistic because early stopping uses the validation set. The test set gives the unbiased figure.

## Stability across random seeds

The graph feature models are retrained with three other seeds. The seed controls which rows and features each tree samples, so a stable model should score similarly under each.

In [11]:
rows = []
for seed in [0, 1, 2]:
    for label, use_shortcuts in [("no shortcuts", False), ("with shortcuts", True)]:
        model = fit_gbm(X_train_g, y_train, X_val_g, y_val, use_shortcuts, ALL_NUMERIC, seed=seed)
        scores = model.predict_proba(gbm_frame(X_val_g, use_shortcuts, ALL_NUMERIC))[:, 1]
        metrics = evaluate(y_val, scores, label)
        rows.append({
            "features": label,
            "seed": seed,
            "pr_auc": metrics["pr_auc"],
            "recall_at_1.0%_fpr": metrics["recall_at_1.0%_fpr"],
            "trees": model.best_iteration_,
        })

seed_runs = pd.DataFrame(rows).sort_values(["features", "seed"])
display(seed_runs)
seed_runs.groupby("features")["pr_auc"].agg(["min", "mean", "max"]).round(4)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_s

,features,seed,pr_auc,recall_at_1.0%_fpr,trees
0,no shortcuts,0,0.282166,0.662162,104
2,no shortcuts,1,0.269629,0.669884,107
4,no shortcuts,2,0.273508,0.670849,128
1,with shortcuts,0,0.446111,0.770270,176
3,with shortcuts,1,0.446709,0.769305,183
5,with shortcuts,2,0.443732,0.771236,176


,min,mean,max
features,,,
no shortcuts,0.2696,0.2751,0.2822
with shortcuts,0.4437,0.4455,0.4467


**Finding:** Results are stable across seeds. Without the shortcut columns, PR-AUC ranges from 0.2696 to 0.2822 over three further seeds (mean 0.2751), with recall at a 1% false positive rate between 66% and 67% and 104 to 128 trees. With the shortcut columns, PR-AUC ranges from 0.4437 to 0.4467 (mean 0.4455), with recall at a 1% false positive rate of 77% and 176 to 183 trees. The main results, trained with seed 42, fall inside both ranges. This shows the scores do not depend on the random sampling during training. It does not show how the model performs on other days of data, which the test set addresses.

## Permutation importance

Each feature is shuffled in the validation set, three times, and the average drop in PR-AUC is recorded. A large drop means the model relies on that feature. This replaces an earlier gain based table, which was distorted by the unstable training run.

In [12]:
def permutation_importance(model, frame, y, n_repeats=3, seed=0):
    rng = np.random.default_rng(seed)
    base = average_precision_score(y, model.predict_proba(frame)[:, 1])
    work = frame.copy()
    rows = []
    for col in frame.columns:
        original = work[col].copy()
        drops = []
        for _ in range(n_repeats):
            work[col] = frame[col].iloc[rng.permutation(len(frame))].values
            drops.append(base - average_precision_score(y, model.predict_proba(work)[:, 1]))
        work[col] = original
        rows.append({
            "feature": col,
            "pr_auc_drop": np.mean(drops),
            "share_of_base": np.mean(drops) / base,
        })
    table = pd.DataFrame(rows).set_index("feature").sort_values("pr_auc_drop", ascending=False)
    return base, table


for name, use_shortcuts in [
    ("LightGBM + graph, no shortcut columns", False),
    ("LightGBM + graph, with format and currency", True),
]:
    frame = gbm_frame(X_val_g, use_shortcuts, ALL_NUMERIC)
    base, table = permutation_importance(models[name], frame, y_val)
    print(f"{name} | baseline PR-AUC: {base:.4f}")
    display(table.round(4))

LightGBM + graph, no shortcut columns | baseline PR-AUC: 0.2702


,pr_auc_drop,share_of_base
feature,,
pair_prior_3d,0.2678,0.9908
amount_vs_currency,0.2162,0.7999
self_transfer,0.1940,0.7177
receiver_in_1d,0.1338,0.4952
log_amount,0.0755,0.2793
sender_in_3d,0.0710,0.2627
receiver_in_3d,0.0694,0.2567
receiver_out_3d,0.0365,0.1352
hour_cos,0.0157,0.0580


LightGBM + graph, with format and currency | baseline PR-AUC: 0.4464


,pr_auc_drop,share_of_base
feature,,
pair_prior_3d,0.4320,0.9676
payment_format,0.3976,0.8906
self_transfer,0.3439,0.7705
amount_vs_currency,0.3250,0.7281
receiver_in_1d,0.1471,0.3296
sender_in_3d,0.1084,0.2427
receiver_in_3d,0.1045,0.2340
receiver_out_3d,0.0521,0.1168
payment_currency,0.0245,0.0549


In [13]:
v = val.join(G)
v["new_pair"] = (v["pair_prior_3d"] == 0) & (~v["self_transfer"])

print(
    v.groupby("new_pair")["is_laundering"]
    .agg(rows="size", laundering="sum", rate="mean")
)

rule = v["new_pair"].to_numpy().astype(int)
tp = int(((rule == 1) & (y_val == 1)).sum())
fp = int(((rule == 1) & (y_val == 0)).sum())
print(f"\nrule: flag every transfer to a new counterparty")
print(f"alerts: {tp + fp:,}   precision: {tp / (tp + fp):.2%}   recall: {tp / y_val.sum():.2%}")
print(evaluate(y_val, rule, "New counterparty rule"))

            rows  laundering      rate
new_pair                              
False     920963         194  0.000211
True       44561         842  0.018895

rule: flag every transfer to a new counterparty
alerts: 44,561   precision: 1.89%   recall: 81.27%
{'model': 'New counterparty rule', 'pr_auc': 0.015558037313458095, 'recall_at_0.1%_fpr': 0.0, 'recall_at_1.0%_fpr': 0.0}


**Finding:** One feature dominates. Shuffling `pair_prior_3d`, the number of earlier transfers between the same sender and receiver in the previous 3 days, removes 99% of PR-AUC without the shortcut columns and 97% with them. The model has learned that laundering goes to counterparties the sender has not paid recently. `self_transfer` (72% drop) acts as a tie breaker, because self transfers also have a pair count of zero. Amount relative to the currency norm matters (80%) far more than raw log amount (28%), and receiver activity matters (`receiver_in_1d` 50%) while the sender's outgoing counts contribute nothing, against the expectation that fan-out would show up there. Where available, `payment_format` is heavily used (89%) and `payment_currency` hardly at all (5%), so payment format is the real shortcut. Hour of day has little effect.

A rule that flags every transfer to a new counterparty confirms this. Such transfers are 4.6% of validation rows but contain 842 of 1,036 laundering transactions, a rate of 1.89% against 0.02% elsewhere. The rule reaches 81.3% recall at 1.89% precision with 44,561 alerts, against 85.9% recall at 0.76% precision and 116,910 alerts for the ACH rule, and has a PR-AUC of 0.0156. It replaces the ACH rule as the baseline to beat. The graph model without shortcut columns scores 0.2702, about 17 times higher, because it ranks these transfers using amount and receiver activity.

This signal is likely stronger here than in real data. Only 4.6% of transfers go to a counterparty not paid in the previous 3 days, which is far more repetitive than real customer behaviour. It uses only earlier transactions and no labels, so it is not leakage, but it is a third signal that depends on how the data was generated, alongside payment format and currency.

## Ablation: retraining without the pair count

Permutation importance scrambles a feature the model was trained to expect, which overstates dependence. Here the graph models are retrained without `pair_prior_3d`, to see how much the remaining features recover.

In [14]:
NO_PAIR = [c for c in ALL_NUMERIC if c != "pair_prior_3d"]

run_gbm("LightGBM + graph, no shortcuts, no pair count", X_train_g, X_val_g, False, NO_PAIR)
run_gbm("LightGBM + graph, with shortcuts, no pair count", X_train_g, X_val_g, True, NO_PAIR)

pd.DataFrame(results).tail(4)

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr,trees
5,"LightGBM + graph, no shortcut columns",0.270240,0.323359,0.662162,118.0
6,"LightGBM + graph, with format and currency",0.446412,0.508687,0.770270,195.0
7,"LightGBM + graph, no shortcuts, no pair count",0.104354,0.158301,0.423745,60.0
8,"LightGBM + graph, with shortcuts, no pair count",0.258917,0.362934,0.636100,251.0


**Finding:** Retraining without `pair_prior_3d` shows partial redundancy. Without the shortcut columns, PR-AUC falls from 0.2702 to 0.1044, a 61% drop, against the 99% drop from permutation. The remaining graph features still give about sixteen times the PR-AUC of the model with no graph features (0.0066), and catch 439 of 1,036 laundering transactions (42%) at a 1% false positive rate. With the shortcut columns, PR-AUC falls from 0.4464 to 0.2589. The pair count therefore multiplies PR-AUC by about 2.6 without shortcuts and about 1.7 with them. It is the strongest single feature, but the result does not rest on it alone. Because the pair count likely benefits from the simulator's repetitive normal behaviour, the behaviour only model is best described by a range: 0.10 from account activity counts alone, and 0.27 with the new counterparty signal included.